# SHAPを見るためのノート

# Import Library

In [ ]:
%pip install --force-reinstall "numpy==1.26.4"

In [ ]:
#Initial setup
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    RocCurveDisplay,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import LogisticRegression
import urllib.request
import joblib
import json

# Import dataset

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_val_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data = temp
display(pro_data)

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis = temp
display(pro_data_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_val.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_sample = temp
display(pro_sample)

In [ ]:
fileDir = "./Data/"
fileName = "map_OID_Uni_Sym.csv"

path = os.path.join(fileDir, fileName)
IDdata = pd.read_csv(path)
IDdata.set_index(IDdata.columns[0],inplace=True)
IDdata.index.name = None
display(IDdata)

In [ ]:
fileDir = "./Data/"
fileName = "DEP_list.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
display(temp)
DEPlist = temp

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis= temp
display(pro_data_dis)

In [ ]:
fileDir = "../Proteomics/Data/"
fileName = "CA199.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
temp = temp.sort_values('SampleID')
temp = temp.set_index('SampleID')
ca199_dis= temp
display(ca199_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_sample_dis = temp
display(pro_sample_dis)

# モデル学習

log変換、欠損値の処理、標準化済み

## ①全プロテオミクスver

In [ ]:
X = pro_data
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:

import os
import time

# ==========================================
# 1. 保存先フォルダ
# ==========================================
save_dir = "./SHAP_Data/All"
os.makedirs(save_dir, exist_ok=True)

# 特徴量名を保存
feature_names = X.columns.tolist()

with open(os.path.join(save_dir, "feature_names.json"), "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

# ==========================================
# 2. 5-fold CV
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

aucs = []
best_params_all = []
feature_importances_all = []

fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    fold = i + 1
    print(f"\n========== Fold {fold} ==========")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    param_grid = {
        "n_estimators": [50, 100, 300, 500],
        "max_depth": [None, 3, 5, 10, 15],
        "min_samples_leaf": [1, 2, 4],
        "max_features": [None, "sqrt"],
        "class_weight": ["balanced"]
    }


    model = RandomForestClassifier(
        criterion="gini",
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=123
    )

    modelcv = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch="2*n_jobs",
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"GridSearchCV: {elapsed / 60:.2f} 分")
    print(f"Best score: {modelcv.best_score_:.4f}")
    print(f"Best hyperparameters: {modelcv.best_params_}")

    # ==========================================
    # 3. 最適モデルを保存
    # ==========================================
    best_model = modelcv.best_estimator_

    model_path = os.path.join(
        save_dir, f"rf_model_fold_{fold}.joblib"
    )
    joblib.dump(best_model, model_path)

    # ==========================================
    # 4. テストデータ・正解ラベルを保存
    # ==========================================
    test_data = X_test.copy()
    test_data["__true_label__"] = y_test.to_numpy()

    test_data.to_csv(
        os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
        index=True,
        index_label="SampleID"
    )

    # ==========================================
    # 5. 各foldの予測とAUC
    # ==========================================
    y_prob = best_model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print(f"Test AUC: {score:.4f}")

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {fold}",
        ax=ax
    )

    # 予測結果を保存
    pd.DataFrame({
        "SampleID": X_test.index,
        "y_true": y_test.to_numpy(),
        "y_prob": y_prob
    }).to_csv(
        os.path.join(save_dir, f"predictions_fold_{fold}.csv"),
        index=False
    )

    # ==========================================
    # 6. 最適パラメータを保存
    # ==========================================
    best_params_all.append({
        "fold": fold,
        "best_cv_auc": float(modelcv.best_score_),
        "test_auc": float(score),
        "best_params": modelcv.best_params_
    })

    # ==========================================
    # 7. 特徴量重要度を保存
    # ==========================================
    importance_df = pd.DataFrame({
        "Protein": X.columns,
        "Importance": best_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    importance_df.to_csv(
        os.path.join(save_dir, f"feature_importance_fold_{fold}.csv"),
        index=False
    )

    feature_importances_all.append(
        importance_df.set_index("Protein")["Importance"].rename(f"Fold_{fold}")
    )

    # GridSearchCVの全結果も保存
    pd.DataFrame(modelcv.cv_results_).to_csv(
        os.path.join(save_dir, f"grid_search_results_fold_{fold}.csv"),
        index=False
    )

# ==========================================
# 8. 結果をまとめて保存
# ==========================================
plt.tight_layout()
plt.savefig(
    os.path.join(save_dir, "roc_curves.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print(f"\n5-fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

with open(
    os.path.join(save_dir, "best_parameters.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(best_params_all, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "Fold": range(1, len(aucs) + 1),
    "AUC": aucs
}).to_csv(
    os.path.join(save_dir, "fold_auc.csv"),
    index=False
)

importance_all_df = pd.concat(feature_importances_all, axis=1)
importance_all_df.to_csv(
    os.path.join(save_dir, "feature_importance_all_folds.csv")
)

print(f"\n保存完了: {os.path.abspath(save_dir)}")


In [ ]:
import time
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

# 各foldの予測結果を保存
y_prob_all = np.full(len(y), np.nan)
y_true_all = np.full(len(y), np.nan)
feature_importances = []

# 追加：各foldの評価指標・最適パラメータを保存
fold_metrics = []
best_params_all = []

save_dir = "./RF_Data/RF_status_all"
os.makedirs(save_dir, exist_ok=True)

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # param_grid = {
    #     'n_estimators': [50, 100, 300, 500],
    #     "max_depth": [None, 3, 5, 10, 15],
    #     'min_samples_leaf': [1, 2, 4],
    #     'max_features': [None, 'sqrt'],
    #     'class_weight': ['balanced']
    # }
    param_grid = {
        'n_estimators': [50],
        "max_depth": [10],
        'min_samples_leaf': [4],
        'max_features': ['sqrt'],
        'class_weight': ['balanced']
    }

    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

    modelcv = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch='2*n_jobs',
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"1回のGridSearchCV: {elapsed / 60:.2f} 分")
    print(f"全5 foldの推定時間: {elapsed * 5 / 60:.2f} 分")
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)

    # 追加：各foldの最適パラメータを保存
    best_params_all.append({
        "Fold": i + 1,
        "Best_Params": modelcv.best_params_
    })

    dataframe = pd.DataFrame(
        modelcv.cv_results_
    ).sort_values('mean_test_score', ascending=False)
    display(dataframe)

    model = modelcv.best_estimator_

    # 予測確率とAUC
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    # OOF予測を保存
    y_prob_all[test_index] = y_prob
    y_true_all[test_index] = np.asarray(y_test)

    # 各foldの特徴量重要度
    feature_importances.append(model.feature_importances_)

    # 追加：各foldの評価指標を計算
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    fold_metrics.append({
        "Fold": i + 1,
        "AUC": score,
        "Accuracy": (tp + tn) / (tp + tn + fp + fn),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "False Positive Rate": fp / (fp + tn) if fp + tn else np.nan,
        "False Negative Rate": fn / (fn + tp) if fn + tp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "NPV": tn / (tn + fn) if tn + fn else np.nan
    })

    print('AUC: ', score)

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}",
        ax=ax
    )

print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")
plt.show()

# ==========================================
# 全foldの予測結果を保存
# ==========================================
y_true_all = y_true_all.astype(int)
y_pred_all = (y_prob_all >= 0.5).astype(int)

oof_df = pd.DataFrame({
    "TrueClass": y_true_all,
    "PredictedProbability": y_prob_all,
    "PredictedClass": y_pred_all
}, index=X.index)

oof_df.index.name = "SampleID"
oof_df.to_csv(f"{save_dir}/oof_predictions.csv")

# ==========================================
# 各foldの評価指標と平均・標準偏差を保存
# ==========================================
fold_metrics_df = pd.DataFrame(fold_metrics)

# 各foldの結果
display(fold_metrics_df.round(4))
fold_metrics_df.to_csv(
    f"{save_dir}/metrics_each_fold.csv",
    index=False
)

# 5-foldの平均と標準偏差
metrics_df = pd.DataFrame({
    "Metric": fold_metrics_df.columns.drop("Fold"),
    "Mean": [
        fold_metrics_df[col].mean()
        for col in fold_metrics_df.columns.drop("Fold")
    ],
    "SD": [
        fold_metrics_df[col].std(ddof=1)
        for col in fold_metrics_df.columns.drop("Fold")
    ]
})

display(metrics_df.round(4))
metrics_df.to_csv(f"{save_dir}/metrics.csv", index=False)

# ==========================================
# foldごとの特徴量重要度の平均を保存
# ==========================================
importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": np.mean(feature_importances, axis=0),
    "SD_Importance": np.std(feature_importances, axis=0, ddof=1)
}).sort_values("Importance", ascending=False)

importance_df.to_csv(
    f"{save_dir}/feature_importance.csv",
    index=False
)

display(importance_df.head(20))

# ==========================================
# 各foldの最適パラメータ5つを保存
# ==========================================
with open(f"{save_dir}/best_parameters.json", "w") as f:
    json.dump(
        best_params_all,
        f,
        indent=2,
        default=str
    )

print("結果を保存しました:", save_dir)

### 偽陰性などチェック

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

save_dir = "./RF_Data/RF_status_all"

# ==========================================
# 1. 各foldの評価指標
# ==========================================
fold_metrics_df = pd.read_csv(
    f"{save_dir}/metrics_each_fold.csv"
)

display(fold_metrics_df.round(4))

# ==========================================
# 2. 5-foldの平均・標準偏差
# ==========================================
metrics_df = pd.read_csv(
    f"{save_dir}/metrics.csv"
)

display(metrics_df.round(4))

# ==========================================
# 3. ROC曲線（各foldのOOF予測から作成）
# ==========================================
oof_df = pd.read_csv(
    f"{save_dir}/oof_predictions.csv",
    index_col="SampleID"
)

y_true = oof_df["TrueClass"].to_numpy()
y_prob = oof_df["PredictedProbability"].to_numpy()
y_pred = oof_df["PredictedClass"].to_numpy()

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"OOF ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# ==========================================
# 4. 混同行列
# ==========================================
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0, 1]
)
disp.plot()
plt.title("Confusion Matrix (OOF)")
plt.show()

# ==========================================
# 5. 偽陽性・偽陰性のサンプルを確認
# ==========================================
fp_df = oof_df[
    (oof_df["TrueClass"] == 0) &
    (oof_df["PredictedClass"] == 1)
]

fn_df = oof_df[
    (oof_df["TrueClass"] == 1) &
    (oof_df["PredictedClass"] == 0)
]

print("偽陽性（False Positive）:", len(fp_df), "samples")
display(fp_df)

print("偽陰性（False Negative）:", len(fn_df), "samples")
display(fn_df)

# 必要ならCSVとして個別に保存
fp_df.to_csv(f"{save_dir}/false_positive_samples.csv")
fn_df.to_csv(f"{save_dir}/false_negative_samples.csv")

# ==========================================
# 6. 特徴量重要度
# ==========================================
importance_df = pd.read_csv(
    f"{save_dir}/feature_importance.csv"
)

display(importance_df.head(20))

top_n = 20
top_features = importance_df.head(top_n).sort_values(
    "Importance", ascending=True
)

plt.figure(figsize=(8, 7))
plt.barh(
    top_features["Protein"],
    top_features["Importance"],
    xerr=top_features["SD_Importance"]
)
plt.xlabel("Mean Feature Importance")
plt.ylabel("Protein")
plt.title(f"Top {top_n} Feature Importances")
plt.tight_layout()
plt.show()

# ==========================================
# 7. 各foldの最適パラメータ
# ==========================================
with open(
    f"{save_dir}/best_parameters.json",
    "r"
) as f:
    best_params_all = json.load(f)

best_params_df = pd.DataFrame([
    {
        "Fold": item["Fold"],
        **item["Best_Params"]
    }
    for item in best_params_all
])

display(best_params_df)

### 進行度で分けてみる

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# ==========================================
# 1. 保存済みのOOF予測結果を読み込む
# ==========================================
save_dir = "./RF_Data/RF_status_all"

oof_df = pd.read_csv(
    f"{save_dir}/oof_predictions.csv",
    index_col="SampleID"
)

# ==========================================
# 2. pro_sampleから進行度を結合
# ==========================================
meta = pro_sample.copy()

# SampleIDをインデックスに統一
if "SampleID" in meta.columns:
    meta = meta.set_index("SampleID")

# SampleIDを文字列に統一
oof_df.index = oof_df.index.astype(str)
meta.index = meta.index.astype(str)

# 予測結果とメタデータを結合
roc_df = oof_df.join(
    meta[["progress_group"]],
    how="inner"
)

print("OOF予測サンプル数:", len(oof_df))
print("結合後のサンプル数:", len(roc_df))

# ==========================================
# 3. 進行度をControl / Early / Advancedに分類
# ==========================================
pg = (
    roc_df["progress_group"]
    .astype(str)
    .str.strip()
    .str.lower()
)

stage = pd.Series(
    np.nan,
    index=roc_df.index,
    dtype="object"
)

# Control
stage.loc[pg.isin(["0"])] = "Control"

# Early
stage.loc[pg.isin(["1"])] = "Early"

# Advanced
stage.loc[pg.isin(["2"])] = "Advanced"

# progress_groupが欠損しているControlを補完
if "CaCo_Status" in meta.columns:
    control_ids = meta.index[
        pd.to_numeric(
            meta["CaCo_Status"], errors="coerce"
        ) == 0
    ]
    stage.loc[
        stage.index.isin(control_ids)
    ] = "Control"

roc_df["Stage"] = stage

print("\n進行度別のサンプル数:")
print(roc_df["Stage"].value_counts(dropna=False))

# ==========================================
# 4. Early vs Control / Advanced vs Control
# ==========================================
plt.figure(figsize=(7, 7))

results = []

for group in ["Early", "Advanced"]:

    # 対象群とControlだけを抽出
    subset = roc_df[
        roc_df["Stage"].isin(["Control", group])
    ].copy()

    # 陽性=対象群、陰性=Control
    y_true = (subset["Stage"] == group).astype(int)

    # 第1段階モデルのCancer予測確率
    y_score = subset["PredictedProbability"]


    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)

    plt.plot(
        fpr,
        tpr,
        label=f"{group} vs Control (AUC = {auc:.3f})"
    )

    results.append({
        "Comparison": f"{group} vs Control",
        "N_Control": int((subset["Stage"] == "Control").sum()),
        f"N_{group}": int((subset["Stage"] == group).sum()),
        "AUC": auc
    })

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    label="Chance"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cancer Detection by Progression Group")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ==========================================
# 5. AUCとサンプル数を表示・保存
# ==========================================
results_df = pd.DataFrame(results)
display(results_df.round(4))




In [ ]:
saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_allpro.joblib'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

In [ ]:

tempA = model.predict(X_test)
tempS = pd.Series(tempA, index = X_test.index, name = 'PredictedClass')
dataframe = y_test.rename('TrueClass')
dataframe = pd.merge(tempS, dataframe, left_index = True, right_index = True, how = 'left')

dataframe2 = pro_data.copy()
display(dataframe['TrueClass'].value_counts())
display(dataframe['PredictedClass'].value_counts())
display(dataframe.loc[dataframe['TrueClass']!=dataframe['PredictedClass']])


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

# 保存済みモデルの読み込み
path = "./Data/mdl_allpro.joblib"
saved_assets = joblib.load(path)

model = saved_assets["model"]

# データ
X = pro_data_dis.copy()
y = pro_sample_dis["groupset2"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("モデル:", model)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)
import matplotlib.pyplot as plt

# データセット
X = pro_data_dis.copy()
y = pro_sample_dis["groupset2"].copy()

# 保存済みモデルの読み込み
save_dir = "./Data"
name = "mdl_allpro.joblib"
path = os.path.join(save_dir, name)

saved_assets = joblib.load(path)
model = saved_assets["model"]

# 予測
y_pred = model.predict(X)

# 予測結果
print("Accuracy:", accuracy_score(y, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y, y_pred))
print("\nClassification report:")
print(classification_report(y, y_pred))

# ROC-AUC（2クラス分類の場合）
if len(model.classes_) == 2:
    positive_class = 1
    positive_idx = list(model.classes_).index(positive_class)

    y_prob = model.predict_proba(X)[:, positive_idx]
    auc = roc_auc_score(y, y_prob)

    print(f"ROC-AUC: {auc:.4f}")

    RocCurveDisplay.from_predictions(y, y_prob)
    plt.title(f"ROC Curve (AUC = {auc:.3f})")
    plt.show()

# 予測結果を保存
results = pd.DataFrame({
    "Actual": y,
    "Predicted": y_pred
}, index=X.index)

if len(model.classes_) == 2:
    results["Predicted_probability_class1"] = y_prob

display(results.head())

In [ ]:
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
# aucs = []
# fig, ax = plt.subplots()

# for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
#     X_train, X_test = X.iloc[train_index], X.iloc[test_index]
#     y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
#     rf_model = RandomForestClassifier(
#         **grid_search.best_params_,
#         max_features="sqrt",
#         random_state=123,
#         class_weight="balanced",
#         n_jobs=-1
#     )
    
#     rf_model.fit(X_train, y_train)
    
#     print(rf_model)
    
#     y_prob = rf_model.predict_proba(X_test)[:, 1]
    
#     auc = roc_auc_score(y_test, y_prob)
#     aucs.append(auc)
    
#     print("AUC:", auc)
    
#     RocCurveDisplay.from_predictions(
#         y_test,
#         y_prob,
#         name=f"Fold {i+1}", ax=ax
#     )
    
# print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
# plt.show()

# importance_df = pd.DataFrame({
#     "Protein": X.columns,
#     "Importance": model.feature_importances_
# })
# importance_df = importance_df.sort_values(
#     "Importance",
#     ascending=False
# )
# print(importance_df.head(20))

In [ ]:
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
# aucs = []
# fig, ax = plt.subplots()

# for i, (train_index, test_index) in enumerate(skf.split(X, y)):
#     X_train, X_test, y_train, y_test = train_test_split(
#         X,
#         y,
#         test_size=0.3,
#         random_state=123,
#         stratify=y
#     )
    
#     rf_model = RandomForestClassifier(
#         n_estimators=300,
#         max_features="sqrt",
#         random_state=123,
#         class_weight="balanced",
#         n_jobs=-1
#     )
    
#     rf_model.fit(X_train, y_train)
    
#     print(rf_model)
    
#     y_prob = rf_model.predict_proba(X_test)[:, 1]
    
#     auc = roc_auc_score(y_test, y_prob)
    
#     print("AUC:", auc)
    
#     RocCurveDisplay.from_predictions(
#         y_test,
#         y_prob,
#         name=f"Fold {i+1}", ax=ax
#     )
    
# print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
# plt.show()

In [ ]:
# importance_df = pd.DataFrame({
#     "Protein": X.columns,
#     "Importance": rf_model.feature_importances_
# })
# importance_df = importance_df.sort_values(
#     "Importance",
#     ascending=False
# )
# print(importance_df.head(20))

# All + CA19-9

In [ ]:
dep = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]
X = pd.concat([
    ca199[['logvalue']],
    pro_data
], axis=1)

y = pro_sample['CaCo_Status']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:

import os
import time

# ==========================================
# 1. 保存先フォルダ
# ==========================================
save_dir = "./SHAP_Data/Allca19-9"
os.makedirs(save_dir, exist_ok=True)

# 特徴量名を保存
feature_names = X.columns.tolist()

with open(os.path.join(save_dir, "feature_names.json"), "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

# ==========================================
# 2. 5-fold CV
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

aucs = []
best_params_all = []
feature_importances_all = []

fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    fold = i + 1
    print(f"\n========== Fold {fold} ==========")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    param_grid = {
        "n_estimators": [50, 100, 300, 500],
        "max_depth": [None, 3, 5, 10, 15],
        "min_samples_leaf": [1, 2, 4],
        "max_features": [None, "sqrt"],
        "class_weight": ["balanced"]
    }


    model = RandomForestClassifier(
        criterion="gini",
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=123
    )

    modelcv = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch="2*n_jobs",
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"GridSearchCV: {elapsed / 60:.2f} 分")
    print(f"Best score: {modelcv.best_score_:.4f}")
    print(f"Best hyperparameters: {modelcv.best_params_}")

    # ==========================================
    # 3. 最適モデルを保存
    # ==========================================
    best_model = modelcv.best_estimator_

    model_path = os.path.join(
        save_dir, f"rf_model_fold_{fold}.joblib"
    )
    joblib.dump(best_model, model_path)

    # ==========================================
    # 4. テストデータ・正解ラベルを保存
    # ==========================================
    test_data = X_test.copy()
    test_data["__true_label__"] = y_test.to_numpy()

    test_data.to_csv(
        os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
        index=True,
        index_label="SampleID"
    )

    # ==========================================
    # 5. 各foldの予測とAUC
    # ==========================================
    y_prob = best_model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print(f"Test AUC: {score:.4f}")

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {fold}",
        ax=ax
    )

    # 予測結果を保存
    pd.DataFrame({
        "SampleID": X_test.index,
        "y_true": y_test.to_numpy(),
        "y_prob": y_prob
    }).to_csv(
        os.path.join(save_dir, f"predictions_fold_{fold}.csv"),
        index=False
    )

    # ==========================================
    # 6. 最適パラメータを保存
    # ==========================================
    best_params_all.append({
        "fold": fold,
        "best_cv_auc": float(modelcv.best_score_),
        "test_auc": float(score),
        "best_params": modelcv.best_params_
    })

    # ==========================================
    # 7. 特徴量重要度を保存
    # ==========================================
    importance_df = pd.DataFrame({
        "Protein": X.columns,
        "Importance": best_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    importance_df.to_csv(
        os.path.join(save_dir, f"feature_importance_fold_{fold}.csv"),
        index=False
    )

    feature_importances_all.append(
        importance_df.set_index("Protein")["Importance"].rename(f"Fold_{fold}")
    )

    # GridSearchCVの全結果も保存
    pd.DataFrame(modelcv.cv_results_).to_csv(
        os.path.join(save_dir, f"grid_search_results_fold_{fold}.csv"),
        index=False
    )

# ==========================================
# 8. 結果をまとめて保存
# ==========================================
plt.tight_layout()
plt.savefig(
    os.path.join(save_dir, "roc_curves.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print(f"\n5-fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

with open(
    os.path.join(save_dir, "best_parameters.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(best_params_all, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "Fold": range(1, len(aucs) + 1),
    "AUC": aucs
}).to_csv(
    os.path.join(save_dir, "fold_auc.csv"),
    index=False
)

importance_all_df = pd.concat(feature_importances_all, axis=1)
importance_all_df.to_csv(
    os.path.join(save_dir, "feature_importance_all_folds.csv")
)

print(f"\n保存完了: {os.path.abspath(save_dir)}")


## ②DEP listのもののみver

In [ ]:
display(DEPlist)

In [ ]:
X = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]
y = pro_sample["CaCo_Status"]

print(DEPlist.shape)
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:

import os
import time

# ==========================================
# 1. 保存先フォルダ
# ==========================================
save_dir = "./SHAP_Data/DEP"
os.makedirs(save_dir, exist_ok=True)

# 特徴量名を保存
feature_names = X.columns.tolist()

with open(os.path.join(save_dir, "feature_names.json"), "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

# ==========================================
# 2. 5-fold CV
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

aucs = []
best_params_all = []
feature_importances_all = []

fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    fold = i + 1
    print(f"\n========== Fold {fold} ==========")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    param_grid = {
        "n_estimators": [50, 100, 300, 500],
        "max_depth": [None, 3, 5, 10, 15],
        "min_samples_leaf": [1, 2, 4],
        "max_features": [None, "sqrt"],
        "class_weight": ["balanced"]
    }


    model = RandomForestClassifier(
        criterion="gini",
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=123
    )

    modelcv = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch="2*n_jobs",
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"GridSearchCV: {elapsed / 60:.2f} 分")
    print(f"Best score: {modelcv.best_score_:.4f}")
    print(f"Best hyperparameters: {modelcv.best_params_}")

    # ==========================================
    # 3. 最適モデルを保存
    # ==========================================
    best_model = modelcv.best_estimator_

    model_path = os.path.join(
        save_dir, f"rf_model_fold_{fold}.joblib"
    )
    joblib.dump(best_model, model_path)

    # ==========================================
    # 4. テストデータ・正解ラベルを保存
    # ==========================================
    test_data = X_test.copy()
    test_data["__true_label__"] = y_test.to_numpy()

    test_data.to_csv(
        os.path.join(save_dir, f"test_data_fold_{fold}.csv"),
        index=True,
        index_label="SampleID"
    )

    # ==========================================
    # 5. 各foldの予測とAUC
    # ==========================================
    y_prob = best_model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print(f"Test AUC: {score:.4f}")

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {fold}",
        ax=ax
    )

    # 予測結果を保存
    pd.DataFrame({
        "SampleID": X_test.index,
        "y_true": y_test.to_numpy(),
        "y_prob": y_prob
    }).to_csv(
        os.path.join(save_dir, f"predictions_fold_{fold}.csv"),
        index=False
    )

    # ==========================================
    # 6. 最適パラメータを保存
    # ==========================================
    best_params_all.append({
        "fold": fold,
        "best_cv_auc": float(modelcv.best_score_),
        "test_auc": float(score),
        "best_params": modelcv.best_params_
    })

    # ==========================================
    # 7. 特徴量重要度を保存
    # ==========================================
    importance_df = pd.DataFrame({
        "Protein": X.columns,
        "Importance": best_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    importance_df.to_csv(
        os.path.join(save_dir, f"feature_importance_fold_{fold}.csv"),
        index=False
    )

    feature_importances_all.append(
        importance_df.set_index("Protein")["Importance"].rename(f"Fold_{fold}")
    )

    # GridSearchCVの全結果も保存
    pd.DataFrame(modelcv.cv_results_).to_csv(
        os.path.join(save_dir, f"grid_search_results_fold_{fold}.csv"),
        index=False
    )

# ==========================================
# 8. 結果をまとめて保存
# ==========================================
plt.tight_layout()
plt.savefig(
    os.path.join(save_dir, "roc_curves.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print(f"\n5-fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

with open(
    os.path.join(save_dir, "best_parameters.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(best_params_all, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "Fold": range(1, len(aucs) + 1),
    "AUC": aucs
}).to_csv(
    os.path.join(save_dir, "fold_auc.csv"),
    index=False
)

importance_all_df = pd.concat(feature_importances_all, axis=1)
importance_all_df.to_csv(
    os.path.join(save_dir, "feature_importance_all_folds.csv")
)

print(f"\n保存完了: {os.path.abspath(save_dir)}")


In [ ]:
modelsets_HER_DEP = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 300, 500],
        "max_depth": [None, 5, 10, 15],
        # 'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced']
    }

    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        # class_weight='balanced',
        n_jobs=1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    modelsets_HER_DEP.append(model)


    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_DEPlist'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
X = pro_data_dis.loc[:, pro_data_dis.columns.isin(DEPlist.index)]

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_DEP):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val) 

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)
# 5モデルのAUCの平均と標準偏差
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()


print(f"Ensemble Validation AUC: {auc_mean:.4f}")

# CA-19-9とIL2RAのモデル

## CA-19-9とIL2RAのデータ読み込み

In [ ]:
ca199 = pro_sample[['CA19_9']].copy()
display(ca199)

il2ra = pro_sample[['sIL2R']].copy()
display(il2ra)

In [ ]:
print((ca199['CA19_9'] <= 0).sum())
print((il2ra['sIL2R'] <= 0).sum())

In [ ]:
ca199['logvalue'] = np.log10(ca199['CA19_9'])
il2ra['logvalue'] = np.log10(il2ra['sIL2R'])


## CA-19-9単独

In [ ]:
X = ca199[["logvalue"]]
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
modelsets_HER_ca199 = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300,400, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3,5, 7],
        'min_samples_leaf': [1, 2, 3,4]
    }
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    modelsets_HER_ca199.append(model)

    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_ca199'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
X = ca199_dis[['logvalue']]

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_ca199):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val)

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)

# 5モデルのAUCの平均 ± SD
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()

print(f"Ensemble Validation AUC: {auc_mean:.4f}")

## IL2RA単独

In [ ]:
X = il2ra[["logvalue"]]
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_il2ra'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

## DEPlist + CA19-9

In [ ]:
dep = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]
X = pd.concat([
    ca199[['logvalue']],
    dep
], axis=1)

y = pro_sample['CaCo_Status']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
modelsets_HER_DEP_ca199 = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300,400, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3,5, 7],
        'min_samples_leaf': [1, 2, 3,4]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)

    # 保存

    model = modelcv.best_estimator_
    modelsets_HER_DEP_ca199.append(model)

    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_DEPlist+ca199'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
display(pro_data_dis)

In [ ]:
dep = pro_data_dis.loc[:, pro_data_dis.columns.isin(DEPlist.index)]
X = pd.concat([
    ca199_dis[['logvalue']],
    dep
], axis=1)

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
display(X)

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_DEP_ca199):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val)

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)

# 5モデルのAUCの平均と標準偏差
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()

print(f"Ensemble Validation AUC: {auc_mean:.4f}")